# Pandas GroupBy operations

**Learning Objectives:**
- Understand the split-apply-combine concept
- Apply groupby operations to aggregate data
- Group data by multiple columns
- Visualize grouped data with plots
- Use transform to create new columns based on groups

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

---

## 1. Introduction to split-apply-combine

📚 **Concept**: The split-apply-combine pattern is a fundamental data analysis strategy:
1. **Split**: Divide data into groups based on some criteria
2. **Apply**: Perform a calculation or operation on each group independently
3. **Combine**: Merge the results back together

This pattern is incredibly powerful for analyzing data across different categories (like sales by region, test scores by gender, or performance by year).

### 1.1 Simple example with test scores

Let's start with a simple example: comparing test scores between boys and girls.

In [ ]:
# Create a simple dataset
data = {
    "BoyOrGirl": ["Boy", "Boy", "Boy", "Girl", "Boy", "Girl", 
                   "Boy", "Boy", "Girl", "Girl", "Boy", "Boy"],
    "TestScore": [25, 94, 57, 62, 70, 25, 94, 57, 62, 70, 62, 70]
}

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(data)
df

<!-- INSTRUCTOR: Emphasize that a groupby object is like a collection of mini-DataFrames -->

In [ ]:
# Create a groupby object
group_by_object = df.groupby("BoyOrGirl")

# 💡 **Tip**: A groupby object contains all the data split into groups,
# but hasn't performed any calculations yet

In [ ]:
# Let's see how many groups we have
print(f"Number of groups: {len(list(group_by_object))}")

In [ ]:
# We can peek inside to see the split data
# Each group is a tuple: (group_name, dataframe_for_that_group)
list(group_by_object)

### 1.2 Applying calculations to groups

Now that we've split the data, let's **apply** some calculations to each group.

In [ ]:
# Calculate the mean test score for each group
df.groupby("BoyOrGirl")["TestScore"].mean()

# ✅ **Key Takeaway**: Boys scored an average of 66.1, girls scored 54.8

In [ ]:
# Find the highest test score in each group
df.groupby("BoyOrGirl")["TestScore"].max()

In [ ]:
# Count how many students in each group
df.groupby("BoyOrGirl").size()

# 💡 **Tip**: Use .size() to count rows in each group

### ✅ Quick check

Before proceeding, can you:
- [ ] Explain what split-apply-combine means?
- [ ] Create a groupby object?
- [ ] Calculate the mean of a grouped column?

---

## 2. Working with real data: Olympic medals

Let's practice with a more realistic dataset containing Olympic summer games medal data.

In [ ]:
# Load Olympic medal data
summer = pd.read_csv("summer.csv")
summer.head()

<!-- INSTRUCTOR: Explain the dataset - each row is a medal won at the Summer Olympics -->

In [ ]:
# Let's explore the data
summer.info()

In [ ]:
# Group by country
group_by_country = summer.groupby("Country")

print(f"Number of countries: {len(list(group_by_country))}")

In [ ]:
# We can verify this matches the number of unique countries
summer["Country"].nunique()

💡 **Tip**: The number of groups always equals the number of unique values in the grouping column.

In [ ]:
# Let's look at one country's data - Sweden (SWE)
# Sweden is at index 121 in our list of groups
sweden_data = list(group_by_country)[121][1]
sweden_data

---

## 3. Common groupby operations

Let's explore aggregation operations using a dataset of car characteristics.

In [ ]:
# Load cars dataset
cars = pd.read_csv("cars.csv")
cars.head()

📚 **Concept**: This dataset contains information about cars from different origins (USA, Europe, Japan). Each car has characteristics like miles per gallon (mpg), horsepower, weight, etc.

### 3.1 Calculating means across groups

In [ ]:
# Calculate mean values for all numeric columns, grouped by origin
cars.groupby("origin").mean(numeric_only=True)

# ✅ **Key Takeaway**: Japanese cars have the best fuel efficiency (30.5 mpg average),
# while American cars are heavier and less efficient (20.1 mpg)

### 3.2 Using aggregate functions

The `.agg()` method allows you to apply multiple aggregation functions at once.

In [ ]:
# Get descriptive statistics for all columns by origin
cars.groupby("origin").agg("describe")

In [ ]:
# Get descriptive statistics for just the mpg column
cars.groupby("origin").agg({"mpg": "describe"})

# 💡 **Tip**: Use a dictionary to specify which aggregations to apply to which columns

---

## 4. Grouping by multiple columns

📚 **Concept**: Sometimes you need to group by more than one column. For example, analyzing fuel efficiency by both country of origin AND year.

In [ ]:
# Group by both origin and model year
multi_group = cars.groupby(["origin", "model_year"]).mean(numeric_only=True)
multi_group

In [ ]:
# Notice the index is now a MultiIndex with two levels
multi_group.index

### 4.1 Finding maximum values in multi-column groups

In [ ]:
# Which combination of origin and year had the highest mean horsepower?
max_hp_combo = cars.groupby(["origin", "model_year"]).mean(numeric_only=True)["horsepower"].idxmax()
print(f"Highest average horsepower: {max_hp_combo}")

# ✅ **Key Takeaway**: American cars in 1970 had the highest average horsepower

In [ ]:
# Which combination had the best fuel efficiency?
max_mpg_combo = cars.groupby(["origin", "model_year"]).mean(numeric_only=True)["mpg"].idxmax()
print(f"Best average fuel efficiency: {max_mpg_combo}")

---

## 5. Visualizing grouped data

📚 **Concept**: Plots are excellent for showing trends in grouped data over time.

### 5.1 Preparing data for plotting

⚠️ **Warning**: When grouping by multiple columns for plotting, the order matters! Put the column you want on the x-axis first.

In [ ]:
# Group by year (first) and origin (second)
# This makes year the primary index
grouped_data = cars.groupby(["model_year", "origin"]).mean(numeric_only=True)
grouped_data["horsepower"]

In [ ]:
# Use .unstack() to pivot the data for plotting
# This converts the origin categories into separate columns
hp_by_year = grouped_data["horsepower"].unstack()
hp_by_year

# 💡 **Tip**: .unstack() converts a MultiIndex into columns, making data easier to plot

### 5.2 Creating trend plots

In [ ]:
# Plot horsepower trends over time by origin
hp_by_year.plot(title="Average Horsepower by Year and Origin", ylabel="Horsepower")
plt.show()

# ✅ **Key Takeaway**: Horsepower decreased over time for all regions,
# likely due to fuel efficiency regulations

In [ ]:
# Plot fuel efficiency trends
mpg_by_year = cars.groupby(["model_year", "origin"]).mean(numeric_only=True)["mpg"].unstack()
mpg_by_year.plot(title="Average MPG by Year and Origin", ylabel="Miles per Gallon")
plt.show()

In [ ]:
# Plot weight trends
weight_by_year = cars.groupby(["model_year", "origin"]).mean(numeric_only=True)["weight"].unstack()
weight_by_year.plot(title="Average Weight by Year and Origin", ylabel="Weight (lbs)")
plt.show()

In [ ]:
# Count of cars per year and origin
car_counts = cars.groupby(["model_year", "origin"]).size().unstack()
car_counts.plot(title="Number of Car Models by Year and Origin", ylabel="Count")
plt.show()

---

## 6. Transform: Creating group-based columns

📚 **Concept**: The `.transform()` method is different from `.agg()`. While `.agg()` returns one value per group, `.transform()` returns a value for **every row** in the original DataFrame.

This is useful when you want to add a column that contains group statistics.

In [ ]:
# Let's add a column showing the mean mpg for each car's model year
cars["mean_mpg_by_year"] = cars.groupby("model_year")["mpg"].transform("mean")
cars[["name", "model_year", "mpg", "mean_mpg_by_year"]].head(10)

💡 **Tip**: Notice how every car from 1970 (model_year=70) has the same mean_mpg_by_year value. Transform broadcasts the group statistic to all rows in that group.

### 6.1 Using transform to fill missing values

A common use case for `.transform()` is filling missing values with group-specific statistics instead of the overall mean.

In [ ]:
# Load a version of the cars dataset with missing mpg values
cars_nan = pd.read_csv("cars_nan.csv")
cars_nan.info()

# Notice: mpg has only 276 non-null values (122 missing)

In [ ]:
# Calculate the mean mpg for each model year
group_means = cars_nan.groupby("model_year")["mpg"].transform("mean")
group_means

📚 **Concept**: Instead of filling all missing values with the same number, we can fill them with the mean for their specific year. This preserves year-to-year trends in the data.

In [ ]:
# Fill missing mpg values with the mean for that car's model year
cars_nan["mpg"] = cars_nan["mpg"].fillna(group_means)

# ✅ **Modern Syntax**: No inplace parameter - assign the result back to the column

In [ ]:
# Verify all missing values are filled
cars_nan.info()

# Now mpg has 398 non-null values (all filled)

---

## Summary

### ✅ Key takeaways

1. **Split-apply-combine** is a powerful pattern for analyzing data across categories
2. Use `.groupby()` to split data into groups based on column values
3. Common aggregation methods: `.mean()`, `.max()`, `.min()`, `.sum()`, `.size()`
4. Use `.agg()` to apply multiple aggregation functions at once
5. Group by multiple columns using a list: `.groupby(["col1", "col2"])`
6. Use `.unstack()` to prepare multi-group data for plotting
7. `.transform()` returns values for every row, while `.agg()` returns one value per group
8. Transform is useful for filling missing values with group-specific statistics

### 💡 Best practices

- Always use `numeric_only=True` with `.mean()` when you have mixed data types
- Use modern pandas syntax: assign results instead of using `inplace=True`
- Put the x-axis variable first when grouping by multiple columns for plotting
- Use descriptive variable names to make your analysis clear

---

### Practice exercises

Ready to practice? Head to the **lab** folder to work on exercises that will reinforce these concepts!